# Clase 1 · El sancocho

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo durante la primera mitad de la clase. Ustedes lo siguen, lo corren y me interrumpen. En la segunda mitad cada uno resuelve el gemelo, *El precio de la luz*, con la misma técnica y otros datos.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Convertir una pregunta de la vida diaria en un problema de datos con métrica, vencer una línea base y validar respetando el tiempo. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular el problema, los objetivos y la métrica), 3 (flujo reproducible y trazable), 4 (implementar y evaluar un modelo) y 5 (comunicar a quien no es técnico). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea de código o que escriba una gráfica. Deben poder explicar sin ayuda qué es una línea base, por qué no se barajan los meses y qué significa el error que reportan. |
| **Entregable** | Este cuaderno corrido de principio a fin en su portafolio, con la decisión de la sección 4 escrita por ustedes. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

## 1. Formulación

Imagínense a alguien que vive de vender sancocho en Manizales. Compra en la Galería, cocina para seis por olla y tiene que imprimir la carta con el precio del plato. La carta dura un trimestre. Si el mercado sube y el precio se quedó quieto, el trimestre se lo come el espinazo.

Esa persona no necesita un modelo. Necesita tres respuestas:

1. **Predecir.** ¿Cuánto costará la olla dentro de tres meses?
2. **Explicar.** ¿Qué ingrediente mueve el costo?
3. **Decidir.** ¿Con qué costo debe presupuestar la olla del próximo trimestre para no quedar corta?

**El dato.** El DANE publica cada mes, en el sistema SIPSA, el precio mayorista por kilo de cientos de productos en las centrales de abasto del país. Manizales reporta por la Galería.

**La receta, para seis:** 1,5 kg de espinazo de cerdo, 1 kg de yuca, 1 kg de papa, tres mazorcas (0,9 kg) y cuatro plátanos verdes (1,4 kg).

**La métrica.** Medimos el error porcentual absoluto medio (MAPE) del pronóstico a tres meses. Un MAPE de 6 % significa que, en promedio, el pronóstico se aleja 6 % del costo real. Elegimos un porcentaje porque el dueño del negocio piensa en márgenes, no en pesos sueltos.

### Predicción escrita

Antes de ver un solo dato, escriban aquí su apuesta. Después la comparamos con lo que salga.

- ¿Cuánto costaban los ingredientes de la olla en diciembre de 2024, al por mayor? **Mi apuesta:**
- ¿Qué ingrediente pesa más en el costo? **Mi apuesta:**
- Si yo pronostico «dentro de tres meses costará lo mismo que hoy», ¿de qué tamaño será mi error? **Mi apuesta:**

## 2. Datos

Una sola línea trae el dato desde el DANE. La función descarga los ocho archivos oficiales, los unifica y revisa con una huella (MD5) que el resultado sea idéntico al que usé para preparar la clase. Tarda cerca de medio minuto.

El archivo no está guardado en el repositorio del curso porque el DANE no permite republicarlo. Por eso cada uno lo descarga de la fuente.

In [ ]:
sipsa = descargar("sipsa")
sipsa.head()

In [ ]:
print("Filas:", len(sipsa))
print("Meses:", sipsa["fecha"].min(), "a", sipsa["fecha"].max())
print("Ciudades:", sipsa["ciudad"].nunique(), "| Productos:", sipsa["producto"].nunique())

Nos quedamos con Manizales. Antes de filtrar productos conviene mirar cómo se llaman, porque el DANE no dice «papa»: dice la variedad.

In [ ]:
manizales = sipsa[sipsa["ciudad"] == "Manizales"]
print(manizales["mercado"].unique())
for palabra in ["espinazo", "Yuca", "Papa", "mazorca", "Plátano"]:
    print(palabra, "->", sorted(p for p in manizales["producto"].unique() if palabra.lower() in p.lower()))

Hay varias papas y varios plátanos. Escoger uno es una **decisión de modelación** y va a la bitácora de la sección 4. Tomo las variedades que se venden todos los meses en la Galería.

In [ ]:
receta = {                                  # kilos para seis personas
    "Carne de cerdo, espinazo": 1.5,
    "Yuca ICA": 1.0,
    "Papa capira": 1.0,
    "Chócolo mazorca": 0.9,
    "Plátano dominico hartón verde": 1.4,
}
precios = (manizales[manizales["producto"].isin(receta)]
           .pivot_table(index="fecha", columns="producto", values="precio_kg"))
precios.index = pd.to_datetime(precios.index)
print("Meses:", len(precios), "| Faltantes por producto:")
print(precios.isna().sum())

Ningún mes falta. El costo de la olla es la suma de precio por cantidad.

In [ ]:
costo_por_ingrediente = precios * pd.Series(receta)
olla = costo_por_ingrediente.sum(axis=1).rename("olla")

ax = olla.plot(color="#A8432C", lw=2)
ax.set(title="Costo mayorista de una olla de sancocho para seis, Manizales", xlabel="", ylabel="pesos")
plt.show()
print(f"Enero de 2013: ${olla.iloc[0]:,.0f} | Diciembre de 2024: ${olla.iloc[-1]:,.0f} | Se multiplicó por {olla.iloc[-1] / olla.iloc[0]:.2f}")

Comparen con su apuesta. Ojo con lo que mide esta cifra: es el costo **mayorista** de los ingredientes en pesos corrientes. No incluye gas, mano de obra ni el margen de la tienda.

## 3. Modelado

### 3.1 La línea base

Todo modelo se mide contra el pronóstico más barato posible. Aquí hay dos candidatos que no cuestan nada:

- **Ingenuo:** dentro de tres meses costará lo mismo que hoy.
- **Estacional:** dentro de tres meses costará lo mismo que ese mes del año pasado.

Si un modelo sofisticado no les gana, no sirve, por elegante que sea.

In [ ]:
H = 3                                             # horizonte: tres meses
tabla = pd.DataFrame({"real": olla.shift(-H)})    # lo que queremos adivinar: el costo en t+3
tabla["ingenuo"] = olla                           # lo que sabemos hoy
tabla["estacional"] = olla.shift(12 - H)          # el mismo mes objetivo, un año antes
tabla = tabla.dropna()

def mape(real, pronostico):
    return float((abs(real - pronostico) / real).mean() * 100)

for metodo in ["ingenuo", "estacional"]:
    print(f"MAPE {metodo}: {mape(tabla['real'], tabla[metodo]):.1f} %")

Ya hay una cifra que vencer. Comparen con su tercera apuesta.

### 3.2 Un modelo, y la trampa de barajar

Probemos una regresión lineal que mira el costo de hoy, el de los dos meses anteriores y el del mismo mes objetivo un año atrás.

In [ ]:
X = pd.DataFrame({
    "hoy": olla,
    "hace_1": olla.shift(1),
    "hace_2": olla.shift(2),
    "anio_pasado": olla.shift(12 - H),
})
datos = X.join(olla.shift(-H).rename("real")).dropna()
variables = list(X.columns)
print(len(datos), "meses utilizables, de", datos.index.min().date(), "a", datos.index.max().date())

Ahora viene la decisión que más errores causa en la práctica: **cómo separar entrenamiento y prueba**. Hagámoslo de las dos maneras y comparemos.

- **Barajando:** se toma al azar el 25 % de los meses para probar. Es lo que hace `train_test_split` por defecto.
- **Respetando el tiempo:** se entrena con todo hasta diciembre de 2021 y se prueba con 2022 en adelante.

**Predicción escrita:** ¿cuál de las dos dará un error más pequeño, y por qué? **Mi apuesta:**

In [ ]:
from sklearn.model_selection import train_test_split

# Manera 1: barajando. Se repite 200 veces para que el resultado no dependa de un sorteo con suerte.
barajados = []
for semilla in range(200):
    X_a, X_b, y_a, y_b = train_test_split(datos[variables], datos["real"], test_size=0.25, random_state=semilla)
    barajados.append(mape(y_b, LinearRegression().fit(X_a, y_a).predict(X_b)))
mape_barajado = float(np.mean(barajados))

# Manera 2: respetando el tiempo
corte = "2021-12-31"
entreno, prueba = datos.loc[:corte], datos.loc[corte:].iloc[1:]
modelo = LinearRegression().fit(entreno[variables], entreno["real"])
pronostico = pd.Series(modelo.predict(prueba[variables]), index=prueba.index)
mape_temporal = mape(prueba["real"], pronostico)

print(f"MAPE barajando (promedio de 200 sorteos): {mape_barajado:.1f} %")
print(f"MAPE respetando el tiempo:                 {mape_temporal:.1f} %")

Los dos números miden el mismo modelo, así que la diferencia viene del examen. Al barajar, el modelo se entrena con meses de 2023 y lo examinan con meses de 2022: ya vio el futuro. Ese examen no se parece a lo que le pasará al dueño del negocio, que siempre pronostica hacia adelante. De aquí en adelante solo vale la validación que respeta el tiempo.

### 3.3 El veredicto, con la misma regla para todos

In [ ]:
comparacion = pd.DataFrame({
    "real": prueba["real"],
    "ingenuo": tabla.loc[prueba.index, "ingenuo"],
    "estacional": tabla.loc[prueba.index, "estacional"],
    "regresion": pronostico,
})
resumen = pd.Series({m: mape(comparacion["real"], comparacion[m]) for m in ["ingenuo", "estacional", "regresion"]}).round(1)
print("MAPE en la prueba (2022 a 2024), en %:")
print(resumen.sort_values())

comparacion.index = comparacion.index + pd.DateOffset(months=H)   # se grafica en el mes pronosticado
ax = comparacion["real"].plot(color="black", lw=2, label="real")
comparacion["ingenuo"].plot(ax=ax, color="#E0A526", label="ingenuo")
comparacion["regresion"].plot(ax=ax, color="#A8432C", ls="--", label="regresión")
ax.set(title="Pronóstico a tres meses contra el costo real", xlabel="", ylabel="pesos"); ax.legend(); plt.show()

Lean la tabla con calma antes de opinar. Si la regresión no le gana con claridad al pronóstico ingenuo, la respuesta honesta para el cliente es el pronóstico ingenuo, que además se explica en una frase. Un modelo que empata con la línea base solo agrega complejidad.

### 3.4 Explicar: ¿qué ingrediente mueve el costo?

In [ ]:
participacion = (costo_por_ingrediente.iloc[-1] / olla.iloc[-1] * 100).round(1).sort_values(ascending=False)
print("Participación en el costo, diciembre de 2024 (%):")
print(participacion)

alza = costo_por_ingrediente.iloc[-1] - costo_por_ingrediente.iloc[0]
print("\nDe cada 100 pesos que subió la olla entre 2013 y 2024, cuántos puso cada ingrediente:")
print((alza / alza.sum() * 100).round(1).sort_values(ascending=False))

costo_por_ingrediente.plot.area(lw=0, cmap="YlOrBr_r")
plt.title("Costo de la olla por ingrediente"); plt.xlabel(""); plt.ylabel("pesos"); plt.legend(fontsize=8); plt.show()

Esta explicación es una descomposición contable: dice cuánto pesa cada ingrediente, no por qué subió su precio. Para el porqué harían falta otros datos, como el precio del maíz que come el cerdo o el clima de la cosecha.

## 4. Comunicación

### 4.1 Decidir: ¿con cuánto se presupuesta la olla?

Un pronóstico de un solo número deja al cliente sin saber cuánto puede fallar. Usamos los errores de la prueba para ponerle un margen. La pregunta del cliente es «con qué costo presupuesto para no quedar corto», así que nos interesa cuánto se queda corto el pronóstico en los meses malos.

In [ ]:
mejor = resumen.idxmin()
comparacion_original = comparacion.copy(); comparacion_original.index = prueba.index
subestimacion = (comparacion_original["real"] / comparacion_original[mejor] - 1)   # positivo: el costo real salió más alto
margen = subestimacion.quantile(0.90)

hoy = olla.index[-1]
if mejor == "ingenuo":
    punto = olla.iloc[-1]
elif mejor == "estacional":
    punto = olla.shift(12 - H).iloc[-1]
else:
    punto = float(modelo.predict(X.iloc[[-1]][variables])[0])

print(f"Método elegido: {mejor} (MAPE {resumen[mejor]} %)")
meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
objetivo = hoy + pd.DateOffset(months=H)
print(f"Pronóstico para {meses[objetivo.month - 1]} de {objetivo.year}: ${punto:,.0f}")
print(f"En 9 de cada 10 meses de prueba, el costo real no superó al pronóstico en más de {margen * 100:.1f} %")
print(f"Costo para presupuestar: ${punto * (1 + margen):,.0f}")

### La decisión en tres frases

Escríbanla para alguien que no sabe qué es una regresión. Una frase para lo que recomiendan, una para la evidencia y una para lo que puede salir mal.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Variedades de la receta (papa capira, yuca ICA, plátano dominico hartón) | Otras variedades | Se reportan todos los meses en la Galería |
| Horizonte de tres meses | Un mes, seis meses | Es lo que dura la carta impresa |
| MAPE como métrica | Error en pesos | El cliente piensa en márgenes |
| Corte temporal en diciembre de 2021 | Separación al azar | Al barajar, el modelo ve el futuro |
| Margen con el percentil 90 de los meses de prueba | Intervalo teórico | Sale de errores observados y se explica sin fórmulas |

### 4.3 Riesgos y mantenimiento

Respondan en dos o tres líneas cada una.

- El dato llega hasta diciembre de 2024. ¿Qué habría que hacer para usar esto hoy?
- El precio es mayorista. ¿En qué se equivoca el cliente si lo lee como el precio de la tienda?
- ¿Cada cuánto habría que revisar si el pronóstico ingenuo sigue siendo el mejor?

### 4.4 Procedencia

Lo último que hace un cuaderno serio es decir de dónde salió su dato.

In [ ]:
procedencia()